# Cal3_S2Stereo

A `Cal3_S2Stereo` represents the calibration for a stereo camera rig. It assumes that both cameras in the rig share the same intrinsic calibration parameters but are separated by a horizontal baseline. 

This class inherits from the standard 5-parameter `Cal3_S2` model and adds a sixth parameter, the baseline `b`. The six parameters are, therefore: two focal lengths ($f_x, f_y$), skew ($s$), the principal point ($u_0, v_0$), and the baseline ($b$). The calibration matrix $K$ is identical to that of `Cal3_S2` and represents the intrinsics of the *left* camera:

$$
K = \begin{bmatrix} f_x & s & u_0 \\ 0 & f_y & v_0 \\ 0 & 0 & 1 \end{bmatrix}
$$

The `uncalibrate` and `calibrate` methods of this class behave identically to those in `Cal3_S2`, converting points for a single camera. The baseline parameter `b` is primarily used by other classes like `StereoCamera` to compute the disparity between the left and right images during 3D projection.

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/geometry/doc/Cal3_S2Stereo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install --quiet gtsam-develop

In [ ]:
import gtsam
import numpy as np
from gtsam import Cal3_S2Stereo, Point2, Point3, Pose3, StereoCamera

## Initialization

A `Cal3_S2Stereo` object can be initialized in several ways:

In [ ]:
# Default constructor
cal_default = Cal3_S2Stereo()
print(f"Default constructor:\n{cal_default}\n")

# From individual parameters: fx, fy, s, u0, v0, b
fx, fy, s, u0, v0, b = 1500.0, 1600.0, 0.1, 320.0, 240.0, 0.5
cal_params = Cal3_S2Stereo(fx, fy, s, u0, v0, b)
print(f"From parameters (fx, fy, s, u0, v0, b):\n{cal_params}\n")

# From a 6-vector [fx, fy, s, u0, v0, b]
cal_vector = np.array([1500.0, 1600.0, 0.1, 320.0, 240.0, 0.5])
cal_vec_ctor = Cal3_S2Stereo(cal_vector)
print(f"From a 6-vector [fx, fy, s, u0, v0, b]:\n{cal_vec_ctor}\n")

# From Field-of-View (FOV), width, height, and baseline
fov, w, h, b_fov = 120, 1280, 720, 0.5
cal_fov = Cal3_S2Stereo(fov, w, h, b_fov)
print(f"From FOV, width, height, and baseline:\n{cal_fov}")

## Properties

Since `Cal3_S2Stereo` inherits from `Cal3_S2`, it has access to all of its parent's methods, in addition to its own specific methods.

In [ ]:
fx, fy, s, u0, v0, b = 1500.0, 1600.0, 0.1, 320.0, 240.0, 0.5
cal = Cal3_S2Stereo(fx, fy, s, u0, v0, b)

print("Calibration object properties:")
print(f"fx: {cal.fx()}")
print(f"fy: {cal.fy()}")
print(f"skew: {cal.skew()}")
print(f"u0: {cal.px()}")
print(f"v0: {cal.py()}")
print(f"Baseline: {cal.baseline()}")
print(f"Principal Point: {cal.principalPoint()}")
print(f"Vector [fx, fy, s, u0, v0, b]: {cal.vector()}")
print(f"K matrix:\n{cal.K()}")

## Basic Operations

The `calibrate` and `uncalibrate` methods work identically to their `Cal3_S2` counterparts, converting points for a single camera's perspective. The baseline does not affect these calculations directly but is used in higher-level classes like `StereoCamera`.

### `calibrate()`

The `calibrate(p)` method converts a 2D point `p` from image pixel coordinates $(u, v)$ to normalized image coordinates $(x, y)$.

In [ ]:
fx, fy, s, u0, v0, b = 1000.0, 1000.0, 0.0, 320.0, 240.0, 0.5
cal_model = Cal3_S2Stereo(fx, fy, s, u0, v0, b)

p_pixels = Point2(420, 290)
p_norm = cal_model.calibrate(p_pixels)
print(f"Pixel point: {p_pixels}")
print(f"Calibrated (normalized) point: {p_norm}")

# This method can optionally compute Jacobians.
Dcal_calibrate = np.zeros((2, 6), order='F') # Note shape is 2x6
Dp_calibrate = np.zeros((2, 2), order='F')
_ = cal_model.calibrate(p_pixels, Dcal_calibrate, Dp_calibrate)
print(f"\nJacobian Dcal_calibrate:\n{Dcal_calibrate}")
print(f"\nJacobian Dp_calibrate:\n{Dp_calibrate}")

### `uncalibrate()`

The `uncalibrate(p)` method converts a 2D point `p` from normalized coordinates back to pixel coordinates.

In [ ]:
p_pixels_recovered = cal_model.uncalibrate(p_norm)
print(f"Normalized point: {p_norm}")
print(f"Uncalibrated (pixel) point: {p_pixels_recovered}")

# This method can also optionally compute Jacobians.
Dcal_uncalibrate = np.zeros((2, 6), order='F')
Dp_uncalibrate = np.zeros((2, 2), order='F')
_ = cal_model.uncalibrate(p_norm, Dcal_uncalibrate, Dp_uncalibrate)
print(f"\nJacobian Dcal_uncalibrate:\n{Dcal_uncalibrate}")
print(f"\nJacobian Dp_uncalibrate:\n{Dp_uncalibrate}")

## Manifold Operations

`Cal3_S2Stereo` is a 6-dimensional manifold and supports `retract` and `localCoordinates` to enable optimization. These operations apply updates in the 6-dimensional tangent space.

In [ ]:
print("Original cal_model:")
print(cal_model)

# Retract: Apply a delta to the calibration parameters
delta_vec = np.array([10.0, 20.0, 0.05, 1.0, -1.0, 0.01])
cal_retracted = cal_model.retract(delta_vec)
print(f"\nDelta vector: {delta_vec}")
print("\nRetracted cal_retracted:")
print(cal_retracted)

# Local coordinates: Find the delta between two calibrations
local_coords = cal_model.localCoordinates(cal_retracted)
print("\nLocal coordinates from cal_model to cal_retracted:")
print(local_coords)

## Relationship with `StereoCamera`

The `Cal3_S2Stereo` object is most powerful when used with the `StereoCamera` class. A `StereoCamera` combines a `Pose3` (the pose of the left camera) and a `Cal3_S2Stereo` calibration to project 3D points into the stereo image plane, producing a `StereoPoint2` which contains the left and right image coordinates ($u_L, u_R, v$).

In [ ]:
fx, fy, s, u0, v0, b = 1000, 1000, 0, 640, 360, 0.5
stereo_calibration = Cal3_S2Stereo(fx, fy, s, u0, v0, b)

# Define the pose of the stereo camera rig in the world frame (left camera's pose)
camera_pose = Pose3(gtsam.Rot3.Ypr(-np.pi/2, 0, -np.pi/2), Point3(0, 0, 5.0))

# Create a StereoCamera object
stereo_camera = StereoCamera(camera_pose, stereo_calibration)

# Define a 3D point in the world frame
p_world = Point3(5, 3, 2)

# Project the 3D point into the stereo camera
p_stereo = stereo_camera.project(p_world)

print(f"3D Point in world frame: {p_world}")
print(f"StereoCamera pose:\n{stereo_camera.pose()}")
print(f"Stereo calibration:\n{stereo_camera.calibration()}")
print(f"\nProjected StereoPoint2 (u_L, u_R, v): {p_stereo}")


## Additional Resources

The following resources provide more detailed explanations of camera calibration and stereo vision.

### Video(s)
- ["Camera Calibration"](https://www.youtube.com/watch?v=F3_a-d02-Z0) by Cyrill Stachniss
- ["Stereo Vision"](https://www.youtube.com/watch?v=hQ-p0_2Ax2I) by Cyrill Stachniss

### Source
- [Cal3_S2Stereo.h](https://github.com/borglab/gtsam/blob/develop/gtsam/geometry/Cal3_S2Stereo.h)
- [Cal3_S2Stereo.cpp](https://github.com/borglab/gtsam/blob/develop/gtsam/geometry/Cal3_S2Stereo.cpp)